# 1.2 Gaussian Elimination and Gauss-Jordan Elimination

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 1:** Systems of Linear Equations

---

### What you will learn

- How to construct and interpret **augmented matrices**
- The three **elementary row operations** and the concept of *row equivalence*
- **Partial pivoting** for numerical stability
- **Row-echelon form (REF)** and **Gaussian elimination**
- **Back-substitution** from REF
- **Reduced row-echelon form (RREF)** and **Gauss-Jordan elimination**
- **Homogeneous systems** and nontrivial solutions

### Prerequisites

- Notebook 1.1 (Introduction to Systems of Linear Equations)

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.elimination import (
    swap_rows,
    scale_row,
    add_scaled_row,
    find_pivot,
    forward_eliminate,
    to_ref,
    to_rref,
    back_substitute,
    solve,
)
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

print("All imports successful.")

---

## 2. Motivation --- The Nutrition Problem

A dietitian is designing a meal plan using three foods. Each food provides
different amounts of three essential vitamins per serving:

| | Vitamin C | Vitamin B$_1$ | Vitamin B$_2$ |
|---|---|---|---|
| **Food A** | 10 | 6 | 8 |
| **Food B** | 4 | 2 | 0 |
| **Food C** | 6 | 4 | 2 |
| **Target** | 32 | 20 | 16 |

Let $x$, $y$, and $z$ be the number of servings of Foods A, B, and C respectively.
The system of equations that matches each vitamin to its target is:

$$\begin{cases}
10x + 4y + 6z = 32 \\
6x + 2y + 4z = 20 \\
8x + 0y + 2z = 16
\end{cases}$$

We will use this system as a running example throughout the notebook,
solving it first by Gaussian elimination (forward elimination + back-substitution)
and then by Gauss-Jordan elimination (full RREF).

In [ ]:
nutrition_data = [
    [10, 4, 6, 32],
    [ 6, 2, 4, 20],
    [ 8, 0, 2, 16],
]

nutrition = Matrix.from_lists(nutrition_data)
print("Augmented matrix [A | b]:")
print(nutrition)

---

## 3. Build --- Core Concepts

We build up the full elimination pipeline in ten steps:

1. Augmented matrix construction
2. Elementary row operations and row equivalence
3. Partial pivoting
4. Row-echelon form (REF)
5. Gaussian elimination with partial pivoting
6. Back-substitution from REF
7. Reduced row-echelon form (RREF)
8. Gauss-Jordan elimination
9. Homogeneous systems
10. Solving homogeneous systems

### 3.1 Augmented Matrix

Given a system $A\mathbf{x} = \mathbf{b}$, the **augmented matrix** $[A \mid \mathbf{b}]$
packs the coefficient matrix and the right-hand side into a single matrix.
For our nutrition problem:

$$
[A \mid \mathbf{b}] =
\begin{bmatrix}
10 & 4 & 6 & 32 \\
6 & 2 & 4 & 20 \\
8 & 0 & 2 & 16
\end{bmatrix}
$$

Every row operation we perform on this matrix corresponds to a legitimate
algebraic manipulation of the original equations. The augmented matrix is
simply a compact bookkeeping device.

In [ ]:
aug = Matrix.from_lists([
    [10, 4, 6, 32],
    [ 6, 2, 4, 20],
    [ 8, 0, 2, 16],
])

print(f"Shape: {aug.rows} x {aug.cols}")
print(f"Coefficient part (first 3 cols): {[aug.get_row(i)[:3] for i in range(aug.rows)]}")
print(f"RHS (last col): {aug.get_col(aug.cols - 1)}")
print()
print(aug)

### 3.2 Definition: Elementary Row Operations

> **Definition (Larson, Sec. 1.2).** The following three operations on the rows of
> a matrix are called **elementary row operations**:
>
> 1. **Interchange** two rows: $R_i \leftrightarrow R_j$
> 2. **Multiply** a row by a nonzero constant: $R_i \leftarrow c \cdot R_i,\; c \neq 0$
> 3. **Add** a multiple of one row to another: $R_i \leftarrow R_i + c \cdot R_j$
>
> Two matrices are **row-equivalent** if one can be obtained from the other
> by a finite sequence of elementary row operations.

**Key principle:** Row-equivalent augmented matrices represent systems with
exactly the same solution set. This is why elimination works --- we transform
the system into a simpler, equivalent one without changing the solutions.

In [ ]:
m = aug.copy()
print("Original:")
print(m)

# Operation 1: Swap R1 <-> R3
swap_rows(m, 0, 2)
print("\nAfter R1 <-> R3 (interchange):")
print(m)

# Operation 2: Scale R1 by 1/8
scale_row(m, 0, 1/8)
print("\nAfter R1 <- (1/8) * R1 (scaling):")
print(m)

# Operation 3: Eliminate below pivot using add_scaled_row
# R2 <- R2 + (-6) * R1
add_scaled_row(m, 1, 0, -6)
print("\nAfter R2 <- R2 + (-6)*R1 (row addition):")
print(m)

print("\nAll three operations demonstrated. The resulting matrix is")
print("row-equivalent to the original --- same solution set.")

### 3.3 Partial Pivoting

When performing elimination on column $k$, we look at all entries in that column
from row $k$ downward and select the one with the **largest absolute value** as
the pivot. We then swap that row into the pivot position.

**Why partial pivoting?**

- Avoids division by zero when the current diagonal entry happens to be zero.
- Reduces **round-off error** in floating-point arithmetic by dividing by
  the largest available number.
- Without pivoting, small pivot values amplify errors in subsequent operations.

The `find_pivot(m, col, start_row)` function returns the row index of the best
pivot candidate, or $-1$ if the entire sub-column is zero.

In [ ]:
m = nutrition.copy()
print("Nutrition matrix:")
print(m)
print()

for col in range(min(m.rows, m.cols - 1)):
    pivot_row = find_pivot(m, col, col)
    sub_col = [abs(m[r, col]) for r in range(col, m.rows)]
    print(f"Column {col}: sub-column magnitudes = {sub_col}")
    if pivot_row >= 0:
        print(f"  Best pivot at row {pivot_row} (value = {m[pivot_row, col]})")
    else:
        print(f"  No valid pivot (column is all zeros below row {col})")
    print()

### 3.4 Definition: Row-Echelon Form (REF)

> **Definition (Larson, Sec. 1.2).** A matrix is in **row-echelon form (REF)** if
> it satisfies all of the following:
>
> 1. All rows consisting entirely of zeros are at the **bottom**.
> 2. For each nonzero row, the **leading entry** (first nonzero element from the left)
>    is in a column to the **right** of the leading entry of the row above it.
> 3. All entries **below** a leading entry are zero.

Visually, REF has a "staircase" pattern of leading entries descending from
upper-left to lower-right:

$$
\begin{bmatrix}
\boxed{*} & * & * & * \\
0 & \boxed{*} & * & * \\
0 & 0 & 0 & \boxed{*}
\end{bmatrix}
$$

Note: the leading entries do not have to be 1 in general REF, though
our `to_ref` normalizes them to 1 for convenience.

In [ ]:
def check_ref(matrix):
    """Check whether a matrix satisfies REF properties."""
    results = []
    prev_lead_col = -1
    found_zero_row = False

    for i in range(matrix.rows):
        row = matrix.get_row(i)
        lead_col = -1
        for j, val in enumerate(row):
            if abs(val) > EPSILON:
                lead_col = j
                break

        if lead_col == -1:
            found_zero_row = True
            continue

        if found_zero_row:
            results.append(f"FAIL: Row {i} is nonzero but follows a zero row.")

        if lead_col <= prev_lead_col:
            results.append(
                f"FAIL: Row {i} leading col ({lead_col}) is not right of "
                f"row {i-1} leading col ({prev_lead_col})."
            )

        for r in range(i + 1, matrix.rows):
            if abs(matrix[r, lead_col]) > EPSILON:
                results.append(
                    f"FAIL: Entry ({r},{lead_col}) = {matrix[r,lead_col]:.4f} "
                    f"is below leading entry of row {i} and should be zero."
                )

        prev_lead_col = lead_col

    if not results:
        results.append("PASS: Matrix is in row-echelon form.")

    return results


ref_example = Matrix.from_lists([
    [1, 3, 0, 2],
    [0, 1, 5, 1],
    [0, 0, 1, 4],
])

not_ref = Matrix.from_lists([
    [1, 3, 0, 2],
    [0, 0, 5, 1],
    [0, 2, 1, 4],
])

print("Matrix 1 (should be REF):")
print(ref_example)
for msg in check_ref(ref_example):
    print(" ", msg)

print("\nMatrix 2 (should NOT be REF):")
print(not_ref)
for msg in check_ref(not_ref):
    print(" ", msg)

### 3.5 Gaussian Elimination with Partial Pivoting

**Gaussian elimination** is the algorithm that converts any matrix into REF
using elementary row operations with partial pivoting.

The algorithm proceeds column by column from left to right:

1. **Find pivot:** In the current column, find the entry with the largest
   absolute value (from the current row downward).
2. **Swap:** Move that row into the pivot position.
3. **Scale:** Divide the pivot row so the leading entry becomes 1.
4. **Eliminate below:** Subtract appropriate multiples of the pivot row
   from all rows below to create zeros beneath the pivot.
5. **Advance:** Move to the next column and next row; repeat.

The `to_ref` function does exactly this, returning the pivot positions
and the number of row swaps performed.

In [ ]:
m = nutrition.copy()
print("Original augmented matrix:")
print(m)
print("\n" + "=" * 50)
print("Gaussian Elimination (to_ref, verbose):")
print("=" * 50 + "\n")

pivot_positions, num_swaps = to_ref(m, verbose=True)

print("\n" + "=" * 50)
print("Final REF:")
print(m)
print(f"\nPivot positions: {pivot_positions}")
print(f"Row swaps performed: {num_swaps}")

### 3.6 Back-Substitution with REF

Once we have the matrix in REF, we can read off the solution by
**back-substitution** --- solving from the bottom row upward.

Given the REF:

$$
\begin{bmatrix}
1 & * & * & | & b_1' \\
0 & 1 & * & | & b_2' \\
0 & 0 & 1 & | & b_3'
\end{bmatrix}
$$

1. Row 3 gives $x_3 = b_3'$ directly.
2. Substitute $x_3$ into row 2 to find $x_2$.
3. Substitute $x_2, x_3$ into row 1 to find $x_1$.

This is exactly what `back_substitute` does.

In [ ]:
sol_type, result = back_substitute(m)

print(f"Solution type: {sol_type}")
print()

if sol_type == "unique":
    print(f"x = {result[0]:.4f}")
    print(f"y = {result[1]:.4f}")
    print(f"z = {result[2]:.4f}")
    print()
    print("Interpretation: the dietitian should use")
    print(f"  {result[0]:.1f} serving(s) of Food A,")
    print(f"  {result[1]:.1f} serving(s) of Food B,")
    print(f"  {result[2]:.1f} serving(s) of Food C.")
elif sol_type == "infinite":
    print("Infinitely many solutions:")
    print(result)
else:
    print("System is inconsistent --- no solution exists.")

### 3.7 Definition: Reduced Row-Echelon Form (RREF)

> **Definition (Larson, Sec. 1.2).** A matrix is in **reduced row-echelon form (RREF)**
> if it satisfies all the conditions of REF, plus:
>
> 4. Every leading entry is $1$.
> 5. Every leading $1$ is the **only nonzero entry** in its column.

RREF is unique --- every matrix has exactly one RREF, regardless of the
sequence of row operations used to reach it.

Example:

$$
\text{REF:}\quad
\begin{bmatrix}
1 & 2 & 0 & 3 \\
0 & 1 & 0 & 1 \\
0 & 0 & 1 & 4
\end{bmatrix}
\quad\xrightarrow{\text{eliminate upward}}\quad
\text{RREF:}\quad
\begin{bmatrix}
1 & 0 & 0 & 1 \\
0 & 1 & 0 & 1 \\
0 & 0 & 1 & 4
\end{bmatrix}
$$

In RREF, each variable corresponding to a pivot column can be read off
directly from the rightmost column --- no back-substitution is needed.

In [ ]:
def check_rref(matrix):
    """Check whether a matrix satisfies RREF properties."""
    results = list(check_ref(matrix))
    if any("FAIL" in r for r in results):
        results.append("FAIL: Not in REF, so cannot be in RREF.")
        return results

    results = []
    for i in range(matrix.rows):
        row = matrix.get_row(i)
        lead_col = -1
        for j, val in enumerate(row):
            if abs(val) > EPSILON:
                lead_col = j
                break

        if lead_col == -1:
            continue

        if abs(matrix[i, lead_col] - 1.0) > EPSILON:
            results.append(
                f"FAIL: Leading entry of row {i} is {matrix[i, lead_col]:.4f}, not 1."
            )

        for r in range(matrix.rows):
            if r != i and abs(matrix[r, lead_col]) > EPSILON:
                results.append(
                    f"FAIL: Entry ({r},{lead_col}) = {matrix[r,lead_col]:.4f} "
                    f"should be 0 (only the leading 1 in column {lead_col})."
                )

    if not results:
        results.append("PASS: Matrix is in reduced row-echelon form.")

    return results


rref_yes = Matrix.from_lists([
    [1, 0, 0, 5],
    [0, 1, 0, 3],
    [0, 0, 1, 2],
])

rref_no = Matrix.from_lists([
    [1, 2, 0, 5],
    [0, 1, 0, 3],
    [0, 0, 1, 2],
])

print("Matrix 1 (should be RREF):")
print(rref_yes)
for msg in check_rref(rref_yes):
    print(" ", msg)

print("\nMatrix 2 (REF but NOT RREF --- entry (0,1) is nonzero):")
print(rref_no)
for msg in check_rref(rref_no):
    print(" ", msg)

### 3.8 Gauss-Jordan Elimination

**Gauss-Jordan elimination** extends Gaussian elimination by performing
**upward elimination** after achieving REF. The result is RREF.

Algorithm:

1. First, use Gaussian elimination to reach REF (with leading 1s).
2. Then, working from the **rightmost pivot upward**, eliminate all entries
   **above** each leading 1.

The advantage of RREF over REF is that the solution can be read off
directly --- no back-substitution step is needed.

The tradeoff is computational cost: Gauss-Jordan requires roughly
$\frac{n^3}{2}$ operations versus $\frac{n^3}{3}$ for Gaussian elimination
with back-substitution. In practice, back-substitution is preferred for
solving systems, while RREF is useful for theoretical analysis
(e.g., finding rank, null space structure).

In [ ]:
m = nutrition.copy()
print("Original augmented matrix:")
print(m)
print("\n" + "=" * 50)
print("Gauss-Jordan Elimination (to_rref, verbose):")
print("=" * 50 + "\n")

pivot_positions = to_rref(m, verbose=True)

print("\n" + "=" * 50)
print("Final RREF:")
print(m)
print(f"\nPivot positions: {pivot_positions}")
print()
print("Reading the solution directly from RREF:")
n_vars = m.cols - 1
for row, col in pivot_positions:
    var_name = ['x', 'y', 'z'][col] if col < 3 else f"x{col+1}"
    print(f"  {var_name} = {m[row, n_vars]:.4f}")

### 3.9 Definition: Homogeneous Systems

> **Definition (Larson, Sec. 1.2).** A system of linear equations is **homogeneous**
> if it can be written in the form
>
> $$A\mathbf{x} = \mathbf{0}$$
>
> That is, every equation has $0$ on the right-hand side.

**Key facts about homogeneous systems:**

1. A homogeneous system is **always consistent** because $\mathbf{x} = \mathbf{0}$
   is always a solution. This is called the **trivial solution**.

2. A **nontrivial solution** (one where $\mathbf{x} \neq \mathbf{0}$) exists
   if and only if the system has **free variables**.

3. **Theorem:** If a homogeneous system has more unknowns than equations
   ($n > m$), then it **must** have nontrivial solutions.
   
   *Proof sketch:* The number of pivots $\leq m < n$, so at least one
   variable is free. Setting it to a nonzero value produces a nontrivial solution.

The solution set of a homogeneous system is always a **subspace** ---
we will explore this concept deeply in Chapter 4.

In [ ]:
# A simple homogeneous system: 2 equations, 2 unknowns
# x + y = 0
# x - y = 0
# Only the trivial solution x = y = 0.
homog_trivial = Matrix.from_lists([
    [1,  1, 0],
    [1, -1, 0],
])

sol_type, result = solve(homog_trivial)
print("Homogeneous system (2 eq, 2 unknowns):")
print(homog_trivial)
print(f"Solution type: {sol_type}")
print(f"Result: {result}")
print("This system has only the trivial solution.")
print()

# Verify: the trivial solution satisfies Ax = 0
if sol_type == "unique":
    print(f"Check: x = {result[0]}, y = {result[1]}")
    print(f"  Eq 1: {1*result[0] + 1*result[1]} = 0? {abs(1*result[0] + 1*result[1]) < EPSILON}")
    print(f"  Eq 2: {1*result[0] - 1*result[1]} = 0? {abs(1*result[0] - 1*result[1]) < EPSILON}")

### 3.10 Solving Homogeneous Systems

Consider the homogeneous system with coefficient matrix

$$
A = \begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
$$

and right-hand side $\mathbf{b} = \mathbf{0}$.

This system has 3 unknowns and 3 equations, but the matrix $A$ is
**singular** (its rows are linearly dependent: $R_3 = 2R_2 - R_1$).
After elimination, we will find a zero row, revealing a free variable
and thus guaranteed nontrivial solutions.

Since there are more unknowns than pivot positions, the theorem from
Section 3.9 guarantees nontrivial solutions exist.

In [ ]:
homog = Matrix.from_lists([
    [1, 2, 3, 0],
    [4, 5, 6, 0],
    [7, 8, 9, 0],
])

print("Homogeneous system [A | 0]:")
print(homog)
print()

m_homog = homog.copy()
print("--- RREF (verbose) ---")
pivots = to_rref(m_homog, verbose=True)
print("\nFinal RREF:")
print(m_homog)
print(f"Pivot positions: {pivots}")
print(f"Number of pivots: {len(pivots)}")
print(f"Number of unknowns: {homog.cols - 1}")
print(f"Free variables: {homog.cols - 1 - len(pivots)}")
print()

sol_type, result = solve(homog.copy())
print(f"Solution type: {sol_type}")
if sol_type == "infinite":
    print(f"Free variables: columns {result['free_vars']}")
    print(f"Particular solution: {result['particular']}")
    print(f"Direction vectors:")
    for fv, direction in result['directions'].items():
        print(f"  t * {direction}  (parameter for x{fv+1})")
    print()
    print("General solution: x = particular + t * direction")
    print("For t = 1:")
    x = [result['particular'][i] + 1 * result['directions'][result['free_vars'][0]][i]
         for i in range(3)]
    print(f"  x = {x}")
    print(f"  Check Ax = 0:")
    A_data = [[1,2,3],[4,5,6],[7,8,9]]
    for i, row in enumerate(A_data):
        val = sum(row[j] * x[j] for j in range(3))
        print(f"    Row {i+1}: {val:.6f}")

### Full Implementation: Using `solve()` in One Call

The `solve` function wraps the entire pipeline --- RREF computation and
solution extraction --- into a single call. It returns the solution type
and result, just like our step-by-step approach above.

In [ ]:
nutrition_fresh = Matrix.from_lists([
    [10, 4, 6, 32],
    [ 6, 2, 4, 20],
    [ 8, 0, 2, 16],
])

sol_type, result = solve(nutrition_fresh)

print(f"Solution type: {sol_type}")
print()

if sol_type == "unique":
    labels = ['x (Food A)', 'y (Food B)', 'z (Food C)']
    for label, val in zip(labels, result):
        print(f"  {label} = {val:.4f}")
    print()

    # Verify
    A = [[10, 4, 6], [6, 2, 4], [8, 0, 2]]
    b = [32, 20, 16]
    print("Verification (Ax = b):")
    for i in range(3):
        computed = sum(A[i][j] * result[j] for j in range(3))
        print(f"  Equation {i+1}: {computed:.4f} = {b[i]} ? {abs(computed - b[i]) < EPSILON}")

---

## 4. Verify --- Cross-Check with NumPy

We verify our from-scratch implementations against NumPy's optimized routines.
For each test, we check:

1. REF and RREF structural properties
2. Agreement with NumPy's solution
3. Residual $\|A\mathbf{x} - \mathbf{b}\|$ is near zero

In [ ]:
def verify_ref(matrix):
    """Return True if matrix is in REF."""
    prev_lead_col = -1
    found_zero_row = False

    for i in range(matrix.rows):
        row = matrix.get_row(i)
        lead_col = -1
        for j, val in enumerate(row):
            if abs(val) > EPSILON:
                lead_col = j
                break

        if lead_col == -1:
            found_zero_row = True
            continue

        if found_zero_row:
            return False

        if lead_col <= prev_lead_col:
            return False

        for r in range(i + 1, matrix.rows):
            if abs(matrix[r, lead_col]) > EPSILON:
                return False

        prev_lead_col = lead_col

    return True


def verify_rref(matrix):
    """Return True if matrix is in RREF."""
    if not verify_ref(matrix):
        return False

    for i in range(matrix.rows):
        row = matrix.get_row(i)
        lead_col = -1
        for j, val in enumerate(row):
            if abs(val) > EPSILON:
                lead_col = j
                break

        if lead_col == -1:
            continue

        if abs(matrix[i, lead_col] - 1.0) > EPSILON:
            return False

        for r in range(matrix.rows):
            if r != i and abs(matrix[r, lead_col]) > EPSILON:
                return False

    return True


# --- REF test ---
m_ref_test = nutrition.copy()
to_ref(m_ref_test)
print("REF of nutrition matrix:")
print(m_ref_test)
print(f"Is REF? {verify_ref(m_ref_test)}")
print()

# --- RREF test ---
m_rref_test = nutrition.copy()
to_rref(m_rref_test)
print("RREF of nutrition matrix:")
print(m_rref_test)
print(f"Is REF? {verify_ref(m_rref_test)}")
print(f"Is RREF? {verify_rref(m_rref_test)}")

In [ ]:
# --- Test 1: Nutrition problem --- compare against NumPy ---
A_np = np.array([[10, 4, 6], [6, 2, 4], [8, 0, 2]], dtype=float)
b_np = np.array([32, 20, 16], dtype=float)
x_np = np.linalg.solve(A_np, b_np)

sol_type, result = solve(nutrition.copy())

print("=== Test 1: Nutrition Problem ===")
print(f"Our solution:   {[round(v, 6) for v in result]}")
print(f"NumPy solution: {[round(v, 6) for v in x_np]}")
print(f"Max difference: {max(abs(result[i] - x_np[i]) for i in range(3)):.2e}")
residual = A_np @ np.array(result) - b_np
print(f"Residual norm:  {np.linalg.norm(residual):.2e}")
print()

In [ ]:
# --- Test 2: New system ---
# 2x - y      = 3
# 4x     - z  = 5
# 6x - 2y + z = 4

test2 = Matrix.from_lists([
    [2, -1,  0, 3],
    [4,  0, -1, 5],
    [6, -2,  1, 4],
])

A2_np = np.array([[2, -1, 0], [4, 0, -1], [6, -2, 1]], dtype=float)
b2_np = np.array([3, 5, 4], dtype=float)

sol_type2, result2 = solve(test2.copy())
print("=== Test 2: 2x - y = 3, 4x - z = 5, 6x - 2y + z = 4 ===")
print(f"Solution type: {sol_type2}")

if sol_type2 == "unique":
    x2_np = np.linalg.solve(A2_np, b2_np)
    print(f"Our solution:   {[round(v, 6) for v in result2]}")
    print(f"NumPy solution: {[round(v, 6) for v in x2_np]}")
    print(f"Max difference: {max(abs(result2[i] - x2_np[i]) for i in range(3)):.2e}")
    residual2 = A2_np @ np.array(result2) - b2_np
    print(f"Residual norm:  {np.linalg.norm(residual2):.2e}")
elif sol_type2 == "inconsistent":
    print("System is inconsistent.")
    try:
        x2_np = np.linalg.solve(A2_np, b2_np)
        residual2 = A2_np @ x2_np - b2_np
        print(f"NumPy gives: {x2_np} (residual: {np.linalg.norm(residual2):.2e})")
    except np.linalg.LinAlgError as e:
        print(f"NumPy also fails: {e}")
else:
    print(f"Infinite solutions: {result2}")
print()

In [ ]:
# --- Test 3: Homogeneous system --- verify Ax = 0 ---
homog_test = Matrix.from_lists([
    [1, 2, 3, 0],
    [4, 5, 6, 0],
    [7, 8, 9, 0],
])

sol_type3, result3 = solve(homog_test.copy())

print("=== Test 3: Homogeneous System ===")
print(f"Solution type: {sol_type3}")

if sol_type3 == "infinite":
    A3_np = np.array([[1,2,3],[4,5,6],[7,8,9]], dtype=float)
    free = result3['free_vars']
    particular = np.array(result3['particular'])
    print(f"Particular solution: {result3['particular']}")
    print(f"Free variables: columns {free}")

    for fv in free:
        d = np.array(result3['directions'][fv])
        print(f"  Direction for x{fv+1}: {list(d)}")
        for t_val in [0, 1, -2, 3.5]:
            x_test = particular + t_val * d
            residual = A3_np @ x_test
            norm = np.linalg.norm(residual)
            status = "PASS" if norm < 1e-9 else "FAIL"
            print(f"    t = {t_val:>5}: x = {list(np.round(x_test, 4))}, "
                  f"||Ax|| = {norm:.2e} [{status}]")

---

## 5. Visualize --- Elimination Step Heatmaps

We visualize the augmented matrix at each major elimination step using
a heatmap. Blue indicates negative values, white indicates zero, and
red indicates positive values. The current pivot is highlighted.

In [ ]:
def capture_elimination_steps(matrix):
    """Run elimination and capture the matrix state after each operation.

    Returns a list of (title, matrix_snapshot, pivot_position) tuples.
    """
    m = matrix.copy()
    steps = [("Original", m.copy(), None)]
    pivot_row = 0

    for col in range(m.cols):
        if pivot_row >= m.rows:
            break

        best = find_pivot(m, col, pivot_row)
        if best == -1:
            continue

        if best != pivot_row:
            swap_rows(m, pivot_row, best)
            steps.append((f"Swap R{pivot_row+1} <-> R{best+1}", m.copy(), (pivot_row, col)))

        pivot_val = m[pivot_row, col]
        scale_row(m, pivot_row, 1.0 / pivot_val)
        steps.append((f"Scale R{pivot_row+1} (pivot=1)", m.copy(), (pivot_row, col)))

        for r in range(pivot_row + 1, m.rows):
            if abs(m[r, col]) > EPSILON:
                factor = -m[r, col]
                add_scaled_row(m, r, pivot_row, factor)

        steps.append((f"Eliminate below col {col}", m.copy(), (pivot_row, col)))
        pivot_row += 1

    # Upward elimination (Gauss-Jordan phase)
    pivot_positions = []
    for i in range(m.rows):
        for j in range(m.cols):
            if abs(m[i, j]) > EPSILON:
                pivot_positions.append((i, j))
                break

    for pr, pc in reversed(pivot_positions):
        changed = False
        for r in range(pr - 1, -1, -1):
            if abs(m[r, pc]) > EPSILON:
                factor = -m[r, pc]
                add_scaled_row(m, r, pr, factor)
                changed = True
        if changed:
            steps.append((f"Eliminate above col {pc}", m.copy(), (pr, pc)))

    return steps


steps = capture_elimination_steps(nutrition)
print(f"Captured {len(steps)} elimination steps.")

In [ ]:
n_steps = len(steps)
cols_per_row = 4
n_plot_rows = (n_steps + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(n_plot_rows, cols_per_row,
                         figsize=(4 * cols_per_row, 3.5 * n_plot_rows))
if n_plot_rows == 1:
    axes = [axes]
axes_flat = [ax for row_axes in axes for ax in row_axes]

vmax = max(abs(v) for s in steps for v in s[1].data)
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

for idx, (title, mat, pivot) in enumerate(steps):
    ax = axes_flat[idx]
    data = np.array(mat.to_lists())

    im = ax.imshow(data, cmap='RdBu_r', norm=norm, aspect='auto')

    for i in range(mat.rows):
        for j in range(mat.cols):
            val = mat[i, j]
            color = 'white' if abs(val) > vmax * 0.6 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=9, color=color, fontweight='bold')

    if pivot is not None:
        pr, pc = pivot
        rect = plt.Rectangle((pc - 0.5, pr - 0.5), 1, 1,
                             linewidth=3, edgecolor='gold', facecolor='none')
        ax.add_patch(rect)

    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xticks(range(mat.cols))
    ax.set_xticklabels([f'c{j}' for j in range(mat.cols)], fontsize=8)
    ax.set_yticks(range(mat.rows))
    ax.set_yticklabels([f'R{i+1}' for i in range(mat.rows)], fontsize=8)

    ax.axvline(x=mat.cols - 1.5, color='black', linewidth=2, linestyle='--')

for idx in range(n_steps, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle('Gaussian & Gauss-Jordan Elimination Steps\n(Nutrition Problem)',
             fontsize=14, fontweight='bold', y=1.02)
fig.colorbar(im, ax=axes_flat[:n_steps], shrink=0.6, label='Cell Value',
             pad=0.02)
plt.tight_layout()
plt.show()

---

## 6. Exercises

Test your understanding with the following problems. Write your solutions in the
code cells below.

### Exercise 1 (Easy)

Use `to_rref` to solve the system:

$$\begin{cases}
x + y + z = 6 \\
2x + 3y + z = 10 \\
x + 2y - z = 2
\end{cases}$$

**Steps:**
1. Create the augmented matrix using `Matrix.from_lists`.
2. Call `to_rref` on a copy of the matrix.
3. Read the solution directly from the RREF.
4. Verify by substituting back into the original equations.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Find **all** solutions to the homogeneous system:

$$\begin{cases}
x_1 + 2x_2 - x_3 + x_4 = 0 \\
2x_1 + 4x_2 + x_3 - 2x_4 = 0
\end{cases}$$

**Steps:**
1. Create the $2 \times 5$ augmented matrix (4 unknowns + 1 RHS column of zeros).
2. Use `solve` to find the solution.
3. Express the general solution in **parametric vector form**:
   $\mathbf{x} = t_1 \mathbf{v}_1 + t_2 \mathbf{v}_2 + \cdots$
4. Verify that two specific choices of the free parameters satisfy $A\mathbf{x} = \mathbf{0}$.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Implement a function `num_solutions(A)` that takes a coefficient matrix $A$
(as a list of lists) and determines how many **free variables** the
homogeneous system $A\mathbf{x} = \mathbf{0}$ has.

**Approach:**
1. Form the augmented matrix $[A \mid \mathbf{0}]$.
2. Compute the RREF using `to_rref`.
3. Count the number of pivot columns.
4. The number of free variables = (number of unknowns) $-$ (number of pivots).

**Test** your function on these five matrices:

| Matrix | Expected free vars |
|---|---|
| $\begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$ | 0 |
| $\begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}$ | 1 |
| $\begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix}$ | 1 |
| $\begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$ | 0 |
| $\begin{bmatrix} 1 & 2 & 3 & 4 \end{bmatrix}$ | 3 |

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Augmented matrix** | Compact representation $[A \mid \mathbf{b}]$ that encodes a full linear system |
| **Elementary row operations** | Swap, scale, add-scaled --- produce row-equivalent matrices with the same solution set |
| **Partial pivoting** | Swap in the largest available pivot to improve numerical stability |
| **REF (Row-echelon form)** | Staircase pattern: leading entries descend left-to-right, zeros below |
| **Gaussian elimination** | Forward elimination to reach REF; solve via back-substitution |
| **RREF (Reduced row-echelon form)** | Leading 1s that are the only nonzero entry in their column --- solution reads off directly |
| **Gauss-Jordan elimination** | Forward + backward elimination to reach RREF |
| **Homogeneous systems** | $A\mathbf{x} = \mathbf{0}$; always consistent; nontrivial solutions iff free variables exist |
| **Free variables** | Arise when there are more unknowns than pivots; parametrize infinite solution families |